# Germany and the GSY Framework: Run-Ups, Crashes, and Industry Price Dynamics

This executed Jupyter notebook presents the German industry-level adaptation of Greenwood, Shleifer, and You (2019), *Bubbles for Fama*. The Stata file `Untitled_revised` is the computation engine; this notebook is the transparent reporting and interpretation layer.

The analysis produces five seminar exhibits: a paper-style descriptive table, an event-time figure, the main threshold results, a Footnote-15-style regression table, and a formal crash-probability test.

## Research question and hypotheses

The central question is whether unusually large industry price run-ups in Germany are followed by unusually high crash risk, as in the U.S. and international evidence in Greenwood, Shleifer, and You (2019).

**Hypotheses**

1. Industries satisfying a large run-up criterion should have a higher subsequent probability of a 40% drawdown.
2. The subsequent crash risk may be stronger for higher run-up thresholds.
3. The relationship should not be explained entirely by pre-run-up volatility.
4. Event-time paths should show strong appreciation before the event and divergence between crash and non-crash episodes afterward.

The German results are an adaptation rather than an exact replication because the sample, country, industry composition, currency history, and available variables differ from the U.S. sample.

## Data and methodology

The Stata workflow starts from `comp_global_daily.dta`, keeps German common-equity observations with valid industry classifications and prices, converts DEM observations into EUR at the fixed rate 1 EUR = 1.95583 DEM, and constructs adjusted total-return prices. Daily returns are cleaned for obvious reversal and extreme-return errors.

The unit of analysis is the **industry-month**, not the individual security and not a single German market index. Monthly industry portfolios are value-weighted using lagged market capitalization based on price and shares outstanding. A value-weighted German market portfolio is used to construct net-of-market industry returns.

A strict run-up at threshold X requires: (i) a prior 24-month raw industry return of at least X, (ii) a prior 24-month net-of-market return of at least X, and (iii) a prior 60-month raw return of at least 50%. The signal is lagged one month, and qualifying observations are separated into distinct episodes.

A crash is a maximum drawdown of at least 40% during the following 24 months. The drawdown calculation permits the event-month value to be the initial peak. Pre-run-up annualized volatility is used only for the controlled regression specifications.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

FINAL_FILES = [
    'germany_table1_paper_style.xlsx',
    'germany_event_time_paths.png',
    'germany_event_time_paths.xlsx',
    'germany_table3_main.xlsx',
    'germany_footnote15_regressions.xlsx',
    'germany_crash_test.xlsx',
]

def find_project_dir():
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents]
    for base in [cwd, *cwd.parents]:
        try:
            candidates.extend(p for p in base.iterdir() if p.is_dir())
        except OSError:
            pass
    scored = []
    for candidate in dict.fromkeys(candidates):
        if candidate.is_dir():
            score = sum((candidate / f).exists() for f in FINAL_FILES)
            if score:
                scored.append((score, candidate))
    if not scored:
        raise FileNotFoundError('Could not find the folder containing the final German exhibits.')
    return sorted(scored, key=lambda x: (-x[0], str(x[1])))[0][1]

PROJECT_DIR = find_project_dir()
display(Markdown(f'**Project folder:** `{PROJECT_DIR}`'))
for filename in FINAL_FILES:
    print(f'{filename}: {"FOUND" if (PROJECT_DIR / filename).exists() else "MISSING"}')

def read_excel(filename):
    path = PROJECT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Missing required exhibit: {filename}')
    return pd.read_excel(path)


## Exhibit 1 — Descriptive evidence comparable to Table 1

This exhibit describes the German raw-only 100% run-up episodes and their subsequent outcomes. It is intended to be comparable in spirit to Table 1 of Greenwood, Shleifer, and You. It should be read descriptively: it shows what happened after identified run-ups, not a causal effect.

The number of firms, run-up dates, forward returns, net-of-market returns, and drawdowns help show whether the German episodes resemble the large price increases and subsequent reversals documented in the paper.

In [ ]:
table1 = read_excel('germany_table1_paper_style.xlsx')
display(table1)
print(f'Table 1 contains {len(table1)} rows and {len(table1.columns)} columns.')


**Interpretation guidance.** Positive post-run-up returns do not rule out a crash: a path can rise strongly over 12 or 24 months while still experiencing a large peak-to-trough loss. The drawdown column is therefore central. Because the German sample is small, individual episodes can materially affect the averages.

## Exhibit 2 — Event-time price paths comparable to Figure 1

The event study normalizes each strict 100% run-up episode to one at event month 0 and follows the industry from month −24 to month +48. The figure separates crash episodes, non-crash episodes, the average run-up path, and the German market benchmark.

A pattern resembling the paper would show substantial appreciation before month 0 and a post-event decline for crash episodes relative to non-crash episodes. This is descriptive event-time evidence, not a causal estimate.

In [ ]:
event_time = read_excel('germany_event_time_paths.xlsx')
display(event_time.head(10))
print(f'Event-time observations: {len(event_time)}')
if 'event_month' in event_time.columns:
    print(f'Event month 0 rows: {(event_time["event_month"] == 0).sum()}')
display(Image(filename=str(PROJECT_DIR / 'germany_event_time_paths.png')))


## Exhibit 3 — Main threshold results comparable to Table 3

This is the main result. It compares run-up thresholds of 50%, 75%, 100%, 125%, and 150%. The episode count is the number of distinct identified run-ups, not the number of qualifying months.

The key columns are subsequent raw and net-market crash rates, average 24-month drawdown, and forward 12- and 24-month raw returns. In the original paper, higher run-up thresholds are associated with more severe subsequent crash patterns. The German table should be interpreted by asking whether the same broad gradient appears here, while recognizing that the German episode counts are much smaller.

In [ ]:
table3 = read_excel('germany_table3_main.xlsx')
display(table3)

# Also display the supporting threshold output when available.
supporting = PROJECT_DIR / 'germany_crash_results.xlsx'
if supporting.exists():
    display(Markdown('### Supporting crash-results output'))
    display(pd.read_excel(supporting))


## Exhibit 4 — Footnote-15-style crash regressions

The regression specification models the crash indicator as a function of whether the industry previously exceeded a raw or net-of-market return threshold. The controlled specification adds pre-run-up annualized volatility. Standard errors are clustered by calendar year in the Stata workflow.

The run-up coefficient is interpreted as the difference in crash probability associated with the indicator, conditional on the included specification. A coefficient that becomes smaller after adding volatility would suggest that part of the raw association is related to pre-run-up volatility. Because the strict German sample is small, p-values and coefficient stability should be emphasized rather than isolated point estimates.

In [ ]:
footnote15 = read_excel('germany_footnote15_regressions.xlsx')
display(footnote15)
print(f'Regression rows: {len(footnote15)}')


## Exhibit 5 — Formal crash-probability test

Greenwood, Shleifer, and You describe the conditional crash probability after a 100% run-up as substantially higher than the unconditional probability. This project tests that comparison explicitly with a two-proportion z-test in the Stata output. The table also compares the strict 100% group with non-100% months.

The correct interpretation depends on both the estimated difference and its p-value. A large numerical difference with a non-small p-value should be described as suggestive rather than statistically conclusive, especially because the strict German group contains few episodes.

In [ ]:
crash_test = read_excel('germany_crash_test.xlsx')
display(crash_test)

if {'Comparison', 'Observations', 'Crashes'}.issubset(crash_test.columns):
    summary = crash_test[['Comparison', 'Observations', 'Crashes']].copy()
    summary['Calculated probability (%)'] = summary['Crashes'] / summary['Observations'] * 100
    display(summary)


## Comparison with the U.S. findings

The comparison should focus on patterns rather than mechanically matching the U.S. numbers. The U.S. paper has a much larger industry sample and therefore more precisely estimated threshold-specific crash rates. Germany has fewer industries and fewer distinct run-up episodes, so the German estimates are more sensitive to individual observations.

The relevant questions are: Does Germany also show large run-ups followed by substantial drawdowns? Does the conditional 100% run-up crash probability exceed the unconditional rate? Do crash and non-crash event paths separate after month 0? Does controlling for volatility attenuate the association? These are the substantive comparisons to make in the seminar.

## Reproducibility and limitations

The notebook reads the executed exhibits generated by `Untitled_revised`; it does not recalculate the raw-data pipeline in Python. The intermediate `.dta` files remain useful for reproducibility, but the five exhibits above are the main seminar outputs.

Important limitations are the small German episode sample, the use of a single WRDS source, the fixed DEM/EUR conversion, the absence of an external risk-free series, possible industry-composition differences relative to the U.S. paper, and the descriptive nature of the event-study evidence. Results should therefore be presented as German evidence consistent with or different from the GSY pattern, not as a causal estimate.

## Conclusion

The German adaptation applies the GSY run-up and crash framework at the industry level. The final conclusion should be written directly from the regenerated exhibits: summarize the number of identified episodes, compare unconditional and conditional crash probabilities, describe the event-time paths, report whether the regression relationship survives the volatility control, and explain how the German evidence compares with the larger U.S. findings.

Do not replace the values in the exhibits with numbers from the earlier project description. The regenerated Stata outputs are the authoritative results.